In [ ]:
import sys
print(sys.executable)

import pinocchio as pin
from pinocchio.robot_wrapper import RobotWrapper

# TODO
# get the path of the dualarm robot
mesh_path = 
urdf = 

model = pin.buildModelFromUrdf(urdf)
data = model.createData()

print("DoF:", model.nq, model.nv)

SyntaxError: invalid syntax (3749050172.py, line 9)

In [ ]:
import mujoco
import mujoco.viewer

import numpy as np

mjcf_path = 
model_mj = mujoco.MjModel.from_xml_path(mjcf_path)
data_mj = mujoco.MjData(model_mj)

mujoco.mj_forward(model_mj, data_mj)
print("MuJoCo nq =", model_mj.nq, " nv =", model_mj.nv, " nu =", model_mj.nu)
print("MuJoCo qpos =", data_mj.qpos)

viewer = mujoco.viewer.launch_passive(model_mj, data_mj)


In [ ]:
# pinocchio model states
joint_indices = np.arange(0, model.nq)

joint_initial_pos = [0.0, 0.0, -0.7854, 0.0, -2.35621, 0.0, 1.5708, 0.785398, 
                          0.0, -0.7854, 0.0, -2.35621, 0.0, 1.5708, 0.785398]
data_mj.qpos[joint_indices] = joint_initial_pos
mujoco.mj_forward(model_mj, data_mj)
print("MuJoCo qpos after setting initial pos =", data_mj.qpos)

# The number of pin_joints should match the number of mujoco joints
assert model.nq == model_mj.nq, "Mismatch in DoF between Pinocchio and MuJoCo models"
def get_pin_state_from_mujoco():
    q  = data_mj.qpos[joint_indices].copy()
    dq = data_mj.qvel[joint_indices].copy()
    return q, dq

mujoco.mj_step(model_mj, data_mj)
viewer.sync()

# Joint Space Impedance Controller using M(q)
$$ M(q)\ddot{q}_{d}​+C(q,\dot{q}​)\dot{q}_{d}​+g(q)-K_{p}e-K_{d}\dot{e}$$

In [ ]:
# Computing Dynamics in Pinocchio
def compute_pin_dynamics(q, dq):
    # Mass Matrix
    M = pin.crba(model, data, q)
    # print("Pinocchio Mass Matrix:\n", M)
    # numerical symmetrization
    M = 0.5 * (M + M.T)
    # print("Symmetrized Mass Matrix:\n", M)

    # Non-linear effects
    nle = pin.rnea(model, data, q, dq, np.zeros_like(dq))
    # print("Pinocchio Non-linear effects:\n", nle)

    return M, nle

In [ ]:
# impedance control law

# Gains Kp and Kd
Kp = np.diag([100.0] * model.nq)
Kd = np.diag([20.0] * model.nq)

def impedance_control(q, dq, q_des, dq_des, ddq_des, M, nle):
    if ddq_des is None:
        ddq_des = np.zeros_like(q)

    # error terms
    e = q_des - q
    de = dq_des - dq

    # Pinocchio impedance control law
    M, nle = compute_pin_dynamics(q, dq)

    # Joint torques
    tau = M @ (ddq_des + Kd @ de + Kp @ e) + nle

    return tau

In [ ]:
# Reference trajectories
mujoco.mj_forward(model_mj, data_mj)
q0, dq0 = get_pin_state_from_mujoco()
print("Initial Pinocchio q0:", q0)
print("Initial Pinocchio dq0:", dq0)

q_goal = q0.copy()
q_goal[1] += 0.5  # Move first joint by 0.5 rad
q_goal[8] -= 0.5  # Move seventh joint by -0.5 rad

T_move = 4.0  # seconds

def desired_trajectory(t):
    if t >= T_move:
        return q_goal, np.zeros_like(q_goal), np.zeros_like(q_goal)
    else:
        s = t / T_move
        q_des = (1 - s) * q0 + s * q_goal
        dq_des = (q_goal - q0) / T_move
        ddq_des = np.zeros_like(q0)
    return q_des, dq_des, ddq_des



In [ ]:
DT = model_mj.opt.timestep
sim_time = 7.0
steps = int(sim_time / DT)

log_t = []
log_q = []
log_dq = []
log_left_ee_pos = []
log_left_ee_vel = []

log_right_ee_pos = []
log_right_ee_vel = []

t = 0.0

for k in range(steps):

    # Get state from Mujoco
    q, dq = get_pin_state_from_mujoco()

    # Desired
    q_des, dq_des, ddq_des = desired_trajectory(t)

    # get dynamics
    M, nle = compute_pin_dynamics(q, dq)

    # Compute torque from Pinocchio impedance
    tau = impedance_control(q, dq, q_des, dq_des, ddq_des, M, nle)

    # Apply to Mujoco
    data_mj.ctrl[:] = tau

    # Step simulation
    mujoco.mj_step(model_mj, data_mj)
    viewer.sync()

    # Log
    log_t.append(t)
    log_q.append(q.copy())
    log_dq.append(dq.copy())


    t += DT

log_q = np.array(log_q)

Until here, then just tests.

In [ ]:
# for i, f in enumerate(model.frames):
#     print(i, f.name)


ee_frame = model.getFrameId("lewis_fr3_link7")
print("Left EE Frame ID:", ee_frame)

# ee_frame = model.getFrameId("lewis_fr3_link1")
# print("Left EE Frame ID:", ee_frame)

In [ ]:
# check the kinematics of the model in the Pinocchio and Mujoco after the simulation
# pinocchio
q = log_q[500]
dq = log_dq[500]

pin.forwardKinematics(model, data, q, dq)
# pin.updateFramePlacements(model, data) 
pin.updateFramePlacements(model, data)
log_left_ee_pos = data.oMf[ee_frame]
log_left_ee_vel = pin.getFrameVelocity(model, data, ee_frame, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)

print(log_left_ee_pos)
print(log_left_ee_vel)

print("np.cross(R@w): ", log_left_ee_pos.rotation@log_left_ee_vel.angular)
pin_w_world = log_left_ee_pos.rotation @ log_left_ee_vel.angular
print("Expected vel in local frame:", log_left_ee_vel.linear - np.cross(log_left_ee_vel.angular, log_left_ee_pos.translation))


In [ ]:
# mujoco
ee_frame_id = mujoco.mj_name2id(model_mj, mujoco.mjtObj.mjOBJ_BODY, "lewis_fr3_link7")

# ee_frame_id = model_mj.body("lewis_fr3_link8").id
print("MuJoCo Left EE Body ID:", ee_frame_id)
data_mj.qpos[joint_indices] = q
data_mj.qvel[joint_indices] = dq
print("dq: ", dq)
mujoco.mj_forward(model_mj, data_mj)  
log_left_ee_pos_mj = data_mj.xpos[ee_frame_id]
print("MuJoCo Left EE Position:", log_left_ee_pos_mj)

mujoco.mj_fwdVelocity(model_mj, data_mj)

#---------------------------------------------------------------------------------------------------------#
# allocate Jacobian buffers
nv = model_mj.nv
jacp = np.zeros((3, nv), dtype=np.float64, order="C")
jacr = np.zeros((3, nv), dtype=np.float64, order="C")

# compute Jacobian in world coords
mujoco.mj_jacBody(model_mj, data_mj, jacp, jacr, ee_frame_id)
J = np.vstack((jacp, jacr))        # 6 × nv

# spatial velocity = J * dq
dq_full = data_mj.qvel.copy()
v_spatial = J @ dq_full           # [vx, vy, vz, wx, wy, wz] in WORLD

v_mj = v_spatial[:3]
w_mj = v_spatial[3:]

print("MuJoCo linear vel:", v_mj)
print("MuJoCo angular vel:", w_mj)

vel_pin = pin.getFrameVelocity(model, data, ee_frame, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
print("Pin linear vel:", vel_pin.linear)
print("Pin angular vel:", vel_pin.angular)

# w_body = mu_left_ee_vel[:3]
# v_body = mu_left_ee_vel[3:]

# # local CoM offset (in body frame)

# # world CoM position
# p_com = mu_left_ee_pos + R @ ipos_local.T
# print("mu_left_ee_pos: ", mu_left_ee_pos)
# print("ipos_local: ", ipos_local)
# print("p_com: ", p_com)

# w_world = w_body
# v_world = v_body

# print("v_body: ", v_body) 
# print("w_world: ", w_world)

In [ ]:
# test the dynamics consistency between Pinocchio and MuJoCo
# we have input the joint positions and velocities from MuJoCo to Pinocchio
# Pinocchio dynamics
M_pin, nle_pin = compute_pin_dynamics(q, dq)
print("Pinocchio Mass Matrix:\n", M_pin)
print("Pinocchio Non-linear effects:\n", nle_pin)

In [ ]:
#TODO: check the consistency between Pinocchio and MuJoCo dynamics

# MuJoCo dynamics
nv = model_mj.nv
nM = nM = model_mj.nM      # number of inertia elements

# allocate full mass matrix
M_full = np.zeros((nv, nv), dtype=np.float64, order="C")

# allocate sparse inertia vector
M_sparse = np.zeros(nM, dtype=np.float64, order="C")

M_sparse[:] = data_mj.qM

# compute full M
# BUG: mujoco.mj_fullM seems to have a bug when there are floating base joints
mujoco.mj_fullM(model_mj, M_full, M_sparse)
h_mj = data_mj.qfrc_bias.copy()  # gravity + coriolis + passive
M_full = 0.5 * (M_full + M_full.T)  # symmetrize

print("MuJoCo Mass Matrix:\n", M_full)
print("MuJoCo Non-linear effects:\n", h_mj)

print("The error in Mass Matrix between Pinocchio and MuJoCo:\n", M_pin - M_full)
print("The error in Non-linear effects between Pinocchio and MuJoCo:\n", nle_pin - h_mj)

In [ ]:
# plotting
import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
plt.plot(log_t, log_q[:,1], label="L joint 1")
plt.plot(log_t, log_q[:,8], label="R joint 1")
plt.grid()
plt.legend()
plt.show()



In [ ]:
link0_rpy = np.array([-0.9, 0.177, -0.15])
lin0_R = pin.rpy.rpyToMatrix(link0_rpy)


link1_pos = np.array([0.0, 0.1, 0.2]) + lin0_R @ np.array([-0.0172, 0.0004, 0.0745])
print(link1_pos)

In [ ]:
from scipy.spatial.transform import Rotation as R

rpy = [-1.570796326794897, 0, 0]
q = R.from_euler('xyz', rpy).as_quat()

print(q)   # [x, y, z, w]